Import Statements


In [ ]:
import os
import re
from typing import List, Dict, Any, Optional, Tuple
from enum import Enum
from dotenv import load_dotenv

# ── NLTK ──────────────────────────────────
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

# ── LangChain Core ────────────────────────
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain.schema import Document

# Load environment variables
load_dotenv()

# Verify LangSmith is configured
print(f"LangSmith Tracing: {os.getenv('LANGCHAIN_TRACING_V2', 'Not set')}")
print(f"LangSmith Project: {os.getenv('LANGCHAIN_PROJECT', 'Not set')}")

In [ ]:
LLM_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
SIMILARITY_THRESHOLD = 0.6
N_RETRIEVAL_RESULTS = 1

In [ ]:
PATIENTS_DB = {
    "P001": {
        "name": "María García",
        "email": "maria.garcia@email.com",
        "phone": "+34 612 345 678",
        "date_of_birth": "1985-03-15",
        "membership_type": "Premium",
        "balance_due": 45.00,
        "upcoming_appointments": [
            {
                "date": "2024-02-15",
                "time": "10:30",
                "doctor": "Dr. Fernández",
                "specialty": "General Medicine",
                "status": "confirmed"
            },
            {
                "date": "2024-03-01",
                "time": "16:00",
                "doctor": "Dr. López",
                "specialty": "Dermatology",
                "status": "pending_confirmation"
            }
        ],
        "last_visit": "2024-01-20",
        "primary_doctor": "Dr. Fernández",
        "allergies": ["Penicillin"],
        "insurance_provider": "Sanitas"
    },
    "P002": {
        "name": "Carlos Rodríguez",
        "email": "carlos.r@email.com",
        "phone": "+34 698 765 432",
        "date_of_birth": "1972-11-28",
        "membership_type": "Basic",
        "balance_due": 0.00,
        "upcoming_appointments": [],
        "last_visit": "2023-11-05",
        "primary_doctor": "Dr. Martín",
        "allergies": [],
        "insurance_provider": "Adeslas"
    },
    "P003": {
        "name": "Ana Belén Torres",
        "email": "anabelen.t@email.com",
        "phone": "+34 655 123 789",
        "date_of_birth": "1990-07-22",
        "membership_type": "Premium",
        "balance_due": 120.50,
        "upcoming_appointments": [
            {
                "date": "2024-02-10",
                "time": "09:00",
                "doctor": "Dr. Sánchez",
                "specialty": "Gynecology",
                "status": "confirmed"
            }
        ],
        "last_visit": "2024-01-28",
        "primary_doctor": "Dr. Sánchez",
        "allergies": ["Ibuprofen", "Latex"],
        "insurance_provider": "Mapfre"
    }
}

In [ ]:
CLINIC_FAQS = {
    "FAQ001": {
        "category": "Appointments",
        "question": "How can I schedule an appointment?",
        "answer": "You can schedule an appointment through our online portal at medicare-plus.com/appointments, by calling our reception at +34 911 234 567 (Monday to Friday, 8:00-20:00), or through this chat by requesting to speak with our scheduling team."
    },
    "FAQ002": {
        "category": "Appointments",
        "question": "What is the cancellation policy?",
        "answer": "Appointments must be cancelled at least 24 hours in advance to avoid a €20 cancellation fee. Premium members can cancel up to 12 hours before without penalty. To cancel, use our online portal or call reception."
    },
    "FAQ003": {
        "category": "Appointments",
        "question": "How early should I arrive for my appointment?",
        "answer": "Please arrive 15 minutes before your scheduled appointment time. First-time patients should arrive 30 minutes early to complete registration paperwork."
    },
    "FAQ004": {
        "category": "Payments",
        "question": "What payment methods do you accept?",
        "answer": "We accept cash, credit/debit cards (Visa, Mastercard, American Express), bank transfers, and payment through major insurance providers (Sanitas, Adeslas, Mapfre, DKV, Asisa). Payment plans are available for treatments exceeding €500."
    },
    "FAQ005": {
        "category": "Payments",
        "question": "How can I check my balance or pay outstanding bills?",
        "answer": "You can view and pay your balance through our patient portal at medicare-plus.com/billing, by phone at +34 911 234 568 (billing department), or in person at our reception desk. We send monthly statements via email for any outstanding balances."
    },
    "FAQ006": {
        "category": "Insurance",
        "question": "Which insurance providers do you work with?",
        "answer": "We work with Sanitas, Adeslas, Mapfre, DKV, and Asisa. Coverage varies by plan. Please bring your insurance card to every visit. We recommend calling your insurance provider to verify coverage before specialized procedures."
    },
    "FAQ007": {
        "category": "Services",
        "question": "What medical specialties are available at the clinic?",
        "answer": "MediCare Plus offers: General Medicine, Pediatrics, Gynecology, Dermatology, Cardiology, Traumatology, Psychology, and Nutrition. Some specialists are only available on specific days. Contact reception for specialist schedules."
    },
    "FAQ008": {
        "category": "Services",
        "question": "Do you offer telemedicine consultations?",
        "answer": "Yes, we offer video consultations for General Medicine, Psychology, Nutrition, and follow-up appointments. Telemedicine is available for Premium members at no extra cost. Basic members pay €15 per video consultation. Book through our portal or call reception."
    },
    "FAQ009": {
        "category": "Hours & Location",
        "question": "What are the clinic's operating hours?",
        "answer": "Regular hours: Monday to Friday 8:00-20:00, Saturday 9:00-14:00. Closed on Sundays and public holidays. Extended hours available by appointment for Premium members."
    },
    "FAQ010": {
        "category": "Hours & Location",
        "question": "Where is the clinic located?",
        "answer": "We are located at Calle Serrano 125, 28006 Madrid. Nearest metro: Núñez de Balboa (Line 5, 9). Paid parking available at Parking Serrano (2-minute walk). Wheelchair accessible entrance on Calle Claudio Coello."
    },
    "FAQ011": {
        "category": "Emergencies",
        "question": "Do you handle medical emergencies?",
        "answer": "MediCare Plus is NOT an emergency facility. For medical emergencies, call 112 or go to the nearest hospital emergency room. For urgent but non-emergency same-day care, Premium members can request urgent appointments subject to availability."
    },
    "FAQ012": {
        "category": "Membership",
        "question": "What is the difference between Basic and Premium membership?",
        "answer": "Basic membership: Standard appointment scheduling, access to all specialists, email support. Premium membership (€29/month): Priority scheduling, 12-hour cancellation policy, free telemedicine, extended hours access, dedicated phone support line, and 10% discount on non-covered services."
    },
    "FAQ013": {
        "category": "Medical Records",
        "question": "How can I access my medical records?",
        "answer": "Access your records through our patient portal at medicare-plus.com/records. You can also request printed copies at reception (€5 processing fee, ready within 48 hours). For records to be sent to another provider, submit a signed authorization form."
    },
    "FAQ014": {
        "category": "Prescriptions",
        "question": "How do I request prescription refills?",
        "answer": "Prescription refills can be requested through the patient portal or by calling your doctor's office directly. Please allow 48-72 hours for processing. Controlled substances require an in-person appointment."
    },
    "FAQ015": {
        "category": "COVID-19",
        "question": "What COVID-19 protocols are in place?",
        "answer": "Masks are optional but recommended in waiting areas. Hand sanitizer stations are available throughout the clinic. If you have COVID-19 symptoms, please call before your visit to arrange appropriate care. Telemedicine is recommended for patients with respiratory symptoms."
    }
}

In [ ]:
class Intent(Enum):
    """Layer 1: Classification of user message intents."""
    EMERGENCY = "emergency"
    PERSONAL_DATA = "personal_data"
    FAQ = "faq"
    UNCLEAR = "unclear"

In [ ]:
# ── NLTK Stopwords ────────────────────────
STOP_WORDS = set(stopwords.words('english'))
print(f"NLTK stopwords loaded: {len(STOP_WORDS)} words")

# ── LangChain LLM ─────────────────────────
llm = ChatOpenAI(
    model=LLM_MODEL,
    temperature=0,
    max_tokens=500
)

# ── LangChain Embeddings ──────────────────
embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL
)

In [ ]:
def build_vectorstore() -> FAISS:
    """
    Convert CLINIC_FAQS into LangChain Documents and build
    a FAISS vector store for semantic retrieval.
    """
    documents = []
    for faq_id, faq in CLINIC_FAQS.items():
        doc = Document(
            page_content=f"Question: {faq['question']}\nAnswer: {faq['answer']}",
            metadata={
                "faq_id": faq_id,
                "category": faq["category"]
            }
        )
        documents.append(doc)

    vectorstore = FAISS.from_documents(
        documents=documents,
        embedding=embeddings
    )
    print(f"Vector store built: {len(documents)} FAQ documents indexed")
    return vectorstore


vectorstore = build_vectorstore()
retriever = vectorstore.as_retriever(
    search_kwargs={"k": N_RETRIEVAL_RESULTS}
)

In [ ]:
def detect_intent(message: str) -> Intent:
    """
    Layer 1: Classify the user message.

    Priority order:
        1. Emergency  (highest — safety first)
        2. Personal   (patient-specific queries)
        3. FAQ        (general clinic information)
        4. Unclear    (fallback)

    Uses NLTK stopwords to clean the message before matching.
    """
    # Clean message: lowercase + remove stopwords
    tokens = re.findall(r'\w+', message.lower())
    cleaned_tokens = [t for t in tokens if t not in STOP_WORDS]
    cleaned = ' '.join(cleaned_tokens)
    original_lower = message.lower()

    # ── Emergency ──────────────────────────
    emergency_keywords = [
        "emergency", "urgent", "severe chest pain", "heart attack",
        "stroke", "bleeding heavily", "unconscious", "can't breathe",
        "not breathing", "dying", "ambulance", "severe pain",
        "chest pain", "having severe", "call 112", "call 911",
        "help me", "critical", "life threatening"
    ]
    if any(kw in original_lower for kw in emergency_keywords):
        return Intent.EMERGENCY

    # ── Personal Data ──────────────────────
    personal_indicators = [
        "my ", "do i owe", "my balance", "my appointment",
        "who is my", "what is my", "when is my", "my doctor",
        "my insurance", "my records", "my prescription",
        "i owe", "my upcoming", "my primary", "my next visit",
        "my allergies", "my last visit", "my membership"
    ]
    if any(ind in original_lower for ind in personal_indicators):
        return Intent.PERSONAL_DATA

    # ── FAQ (semantic similarity) ──────────
    results = vectorstore.similarity_search_with_score(message, k=1)
    if results and results[0][1] < (2 - SIMILARITY_THRESHOLD):
        return Intent.FAQ

    return Intent.UNCLEAR

In [ ]:
def retrieve_patient_data(user_id: str, message: str) -> Optional[str]:
    """
    Layer 2 — Personal branch: Look up PATIENTS_DB.

    Uses NLTK stopwords to clean the message before keyword matching.
    """
    if user_id not in PATIENTS_DB:
        return None

    patient = PATIENTS_DB[user_id]
    name = patient["name"].split()[0]

    # Clean message for keyword matching
    tokens = re.findall(r'\w+', message.lower())
    cleaned_tokens = [t for t in tokens if t not in STOP_WORDS]
    cleaned = ' '.join(cleaned_tokens)

    # ── Balance / Payment ──────────────────
    if any(w in cleaned for w in ["owe", "balance", "bill", "debt", "pay"]):
        balance = patient["balance_due"]
        if balance > 0:
            return f"Hi {name}, you have a balance of €{balance:.2f}."
        return f"Hi {name}, you don't have any outstanding balance."

    # ── Upcoming Appointments ──────────────
    if any(w in cleaned for w in ["next", "visit", "appointment", "upcoming"]):
        appts = patient["upcoming_appointments"]
        if appts:
            a = appts[0]
            return (f"Hi {name}, your next appointment is on {a['date']} "
                    f"at {a['time']} with {a['doctor']} ({a['specialty']}). "
                    f"Status: {a['status']}.")
        return f"Hi {name}, you don't have any upcoming appointments."

    # ── Primary Doctor ─────────────────────
    if any(w in cleaned for w in ["primary", "doctor"]):
        return f"Hi {name}, your primary doctor is {patient['primary_doctor']}."

    # ── Insurance ──────────────────────────
    if "insurance" in cleaned:
        return f"Hi {name}, your insurance provider is {patient['insurance_provider']}."

    # ── Allergies ──────────────────────────
    if "allerg" in cleaned:
        allergies = patient["allergies"]
        if allergies:
            return f"Hi {name}, your recorded allergies are: {', '.join(allergies)}."
        return f"Hi {name}, you have no recorded allergies."

    # ── Last Visit ─────────────────────────
    if "last" in cleaned and "visit" in cleaned:
        return f"Hi {name}, your last visit was on {patient['last_visit']}."

    # ── Membership ─────────────────────────
    if "membership" in cleaned:
        return f"Hi {name}, you have a {patient['membership_type']} membership."

    return None

In [ ]:
def retrieve_faq_answer(message: str) -> Optional[Dict[str, str]]:
    """
    Layer 2 — FAQ branch: Semantic search via LangChain FAISS retriever.
    """
    results = vectorstore.similarity_search_with_score(message, k=1)

    if results:
        doc, score = results[0]
        # FAISS uses L2 distance: lower = more similar
        if score < (2 - SIMILARITY_THRESHOLD):
            faq_id = doc.metadata.get("faq_id", "Unknown")
            return {
                "faq_id": faq_id,
                "answer": doc.page_content,
                "score": score
            }
    return None

In [ ]:
def retrieve_contextual_answer(user_id: str, message: str,
                                 history: List[Dict[str, str]]) -> Optional[str]:
    """
    Layer 2 — Contextual branch: Resolve doctor → specialty from history.
    """
    if "specialty" not in message.lower():
        return None

    doctor_pattern = r'Dr\.\s+\w+'
    mentioned = []
    for turn in history:
        mentioned.extend(
            re.findall(doctor_pattern, turn.get("content", ""), re.IGNORECASE)
        )
    mentioned.extend(re.findall(doctor_pattern, message, re.IGNORECASE))

    if not mentioned or user_id not in PATIENTS_DB:
        return None

    target = mentioned[-1]
    patient = PATIENTS_DB[user_id]

    for appt in patient["upcoming_appointments"]:
        if target.lower() in appt["doctor"].lower():
            return f"{target}'s specialty is {appt['specialty']}."

    return None

In [ ]:
def resolve_references(message: str,
                       conversation_history: List[Dict[str, str]]) -> str:
    """
    Layer 3: Resolve pronouns and references using conversation history.

    Handles: "his", "her", "their", "the same doctor", "that doctor",
             "that specialty"
    """
    if not conversation_history:
        return message

    resolved = message

    # ── Extract entities from history ─────
    doctor_pattern = r'Dr\.\s+\w+'
    specialty_keywords = [
        "general medicine", "pediatrics", "gynecology", "dermatology",
        "cardiology", "traumatology", "psychology", "nutrition"
    ]

    mentioned_doctors = []
    mentioned_specialties = []

    for turn in conversation_history:
        content = turn.get("content", "")
        mentioned_doctors.extend(
            re.findall(doctor_pattern, content, re.IGNORECASE)
        )
        for spec in specialty_keywords:
            if spec in content.lower():
                mentioned_specialties.append(spec)

    # Deduplicate preserving order
    seen = set()
    unique_doctors = []
    for d in mentioned_doctors:
        key = d.lower()
        if key not in seen:
            seen.add(key)
            unique_doctors.append(d)

    # ── Resolve doctor pronouns ───────────
    if unique_doctors:
        last_doctor = unique_doctors[-1]
        replacements = [
            (r'\bhis specialty\b', f"{last_doctor}'s specialty"),
            (r'\bher specialty\b', f"{last_doctor}'s specialty"),
            (r'\btheir specialty\b', f"{last_doctor}'s specialty"),
            (r'\bthe same doctor\b', last_doctor),
            (r'\bthat doctor\b', last_doctor),
            (r'\bhis\b', last_doctor),
            (r'\bher\b', last_doctor),
        ]
        for pattern, replacement in replacements:
            resolved = re.sub(pattern, replacement, resolved, flags=re.IGNORECASE)

    # ── Resolve specialty references ──────
    if mentioned_specialties and "that specialty" in resolved.lower():
        last_spec = mentioned_specialties[-1]
        resolved = re.sub(
            r'\bthat specialty\b', last_spec, resolved, flags=re.IGNORECASE
        )

    return resolved

In [ ]:
def check_escalation(message: str, intent: Intent,
                     rag_result: Optional[str]) -> Tuple[bool, str]:
    """
    Layer 4: Determine if escalation to a human is needed.

    Triggers:
        1. Emergency detected
        2. Negative sentiment (anger/frustration)
        3. Unanswerable query (no RAG match)
    """
    original_lower = message.lower()

    # 1. Emergency
    if intent == Intent.EMERGENCY:
        return True, "emergency"

    # 2. Negative sentiment
    frustration_keywords = [
        "angry", "frustrated", "furious", "unacceptable", "ridiculous",
        "terrible service", "worst", "hate", "useless", "incompetent",
        "waste of time", "fed up", "annoyed", "disappointed",
        "very upset", "so angry", "can't believe"
    ]
    if any(kw in original_lower for kw in frustration_keywords):
        return True, "negative_sentiment"

    # 3. Unanswerable
    if intent == Intent.UNCLEAR:
        return True, "unanswerable"
    if intent in [Intent.FAQ, Intent.PERSONAL_DATA] and rag_result is None:
        return True, "unanswerable"

    return False, "none"

In [ ]:
def format_response(content: str, source: str = None) -> str:
    """Layer 5: Append source citation."""
    if source and "[Source:" not in content:
        return f"{content} [Source: {source}]"
    return content

In [ ]:
def format_escalation(reason: str) -> str:
    """Layer 5: Return appropriate escalation message."""
    messages = {
        "emergency": (
            "🚨 I understand this is an emergency. I'm immediately "
            "transferring you to a human agent. Please stay on the line. "
            "Transferring to agent..."
        ),
        "negative_sentiment": (
            "I'm sorry to hear you're frustrated. Let me connect you with "
            "a human agent who can better assist you. Transferring to agent..."
        ),
        "unanswerable": (
            "I'm sorry, I cannot find information related to your query "
            "in our database. Would you like to speak to a human agent "
            "who can help you further?"
        )
    }
    return messages.get(reason, "Transferring to agent...")

In [ ]:
def generate_response(
    user_id: str,
    current_message: str,
    conversation_history: List[Dict[str, str]]
) -> Dict[str, Any]:
    """
    5-Layer Pipeline:

        Layer 3 → Contextual Memory   (resolve references)
        Layer 1 → Intent Detection    (classify message)
        Layer 2 → RAG Logic           (retrieve answer)
        Layer 4 → Escalation Checks   (decide escalation)
        Layer 5 → Output Formatting   (format + cite)

    All LangChain calls are automatically traced by LangSmith.
    """

    # ── LAYER 3: Contextual Memory ────────
    resolved_message = resolve_references(current_message, conversation_history)

    # ── LAYER 1: Intent Detection ─────────
    intent = detect_intent(resolved_message)

    # ── LAYER 2: RAG Logic ────────────────
    rag_result = None
    source = None

    if intent == Intent.EMERGENCY:
        pass  # No retrieval needed

    elif intent == Intent.PERSONAL_DATA:
        rag_result = retrieve_patient_data(user_id, resolved_message)
        source = "Patient Data"

        # Fallback: contextual doctor → specialty
        if rag_result is None:
            rag_result = retrieve_contextual_answer(
                user_id, resolved_message, conversation_history
            )
            source = "Patient Data"

    elif intent == Intent.FAQ:
        faq_match = retrieve_faq_answer(resolved_message)
        if faq_match:
            rag_result = faq_match["answer"]
            source = faq_match["faq_id"]

    # ── LAYER 4: Escalation Checks ────────
    should_escalate, reason = check_escalation(
        resolved_message, intent, rag_result
    )

    # ── LAYER 5: Output Formatting ────────
    if should_escalate:
        response = format_escalation(reason)
    else:
        response = format_response(rag_result, source)

    return {
        "response": response,
        "escalate_to_human": should_escalate
    }

In [ ]:
if __name__ == "__main__":
    print("\n" + "=" * 60)
    print("  AI CUSTOMER SERVICE BOT — 5-LAYER TEST SUITE")
    print("=" * 60)

    # ── Test 1: Happy Path ────────────────
    print("\n📋 Test 1: The Happy Path")
    print("-" * 40)
    r = generate_response("P001", "How do I pay?", [])
    print(f"  User:  How do I pay?")
    print(f"  Bot:   {r['response']}")
    print(f"  Esc:   {r['escalate_to_human']}")

    # ── Test 2: Personal Query ────────────
    print("\n📋 Test 2: The Personal Query")
    print("-" * 40)
    r = generate_response("P003", "Do I owe anything?", [])
    print(f"  User:  Do I owe anything?")
    print(f"  Bot:   {r['response']}")
    print(f"  Esc:   {r['escalate_to_human']}")

    # ── Test 3: Emergency ─────────────────
    print("\n📋 Test 3: The Emergency")
    print("-" * 40)
    r = generate_response("P002", "I'm having severe chest pain.", [])
    print(f"  User:  I'm having severe chest pain.")
    print(f"  Bot:   {r['response']}")
    print(f"  Esc:   {r['escalate_to_human']}")

    # ── Test 4: Hallucination Trap ────────
    print("\n📋 Test 4: The Hallucination Trap")
    print("-" * 40)
    r = generate_response("P001", "Do you do liver transplants?", [])
    print(f"  User:  Do you do liver transplants?")
    print(f"  Bot:   {r['response']}")
    print(f"  Esc:   {r['escalate_to_human']}")

    # ── Test 5: Memory (Multi-turn) ───────
    print("\n📋 Test 5: Memory (Multi-turn)")
    print("-" * 40)
    history = []

    r1 = generate_response("P001", "Who is my primary doctor?", history)
    print(f"  User (T1): Who is my primary doctor?")
    print(f"  Bot:       {r1['response']}")
    history.append({"role": "user", "content": "Who is my primary doctor?"})
    history.append({"role": "bot",   "content": r1["response"]})

    r2 = generate_response("P001", "What is his specialty?", history)
    print(f"\n  User (T2): What is his specialty?")
    print(f"  Bot:       {r2['response']}")
    print(f"  Esc:       {r2['escalate_to_human']}")

    print("\n" + "=" * 60)
    print("  ✅ TESTING COMPLETE — Check LangSmith for traces!")
    print("=" * 60)